In [1]:
# Import the packages needed to connect to MySQL and export the table.
import pandas as pd

from pathlib import Path
from getpass import getpass
from urllib.parse import quote_plus
from sqlalchemy import create_engine


# Define the location and filename for the exported CSV.
output_path = (
    Path.home()
    / "Downloads"
    / "CFPB_Banking_Project"
    / "cfpb_banking_complaints_clean_2021_2025.csv"
)


# Request the MySQL password without displaying or saving it in the notebook.
mysql_password = quote_plus(
    getpass("Enter your MySQL password: ")
)


# Create a connection to the MySQL database.
# Change 'root' only if you use a different MySQL username.
engine = create_engine(
    f"mysql+pymysql://root:{mysql_password}"
    "@localhost:3306/cfpb_banking_complaints"
)


# Select the complete cleaned table containing all five product groups.
query = """
SELECT *
FROM complaints_clean;
"""


# Export 50,000 rows at a time to avoid loading all
# 888,691 records into computer memory simultaneously.
first_chunk = True
total_exported = 0

with engine.connect().execution_options(stream_results=True) as connection:
    for chunk in pd.read_sql_query(
        query,
        connection,
        chunksize=50_000
    ):
        chunk.to_csv(
            output_path,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False,
            encoding="utf-8"
        )

        total_exported += len(chunk)
        print(f"Exported {total_exported:,} rows")

        first_chunk = False


# Close the database connection resources.
engine.dispose()

print(f"\nExport complete: {output_path}")

Exported 50,000 rows
Exported 100,000 rows
Exported 150,000 rows
Exported 200,000 rows
Exported 250,000 rows
Exported 300,000 rows
Exported 350,000 rows
Exported 400,000 rows
Exported 450,000 rows
Exported 500,000 rows
Exported 550,000 rows
Exported 600,000 rows
Exported 650,000 rows
Exported 700,000 rows
Exported 750,000 rows
Exported 800,000 rows
Exported 850,000 rows
Exported 888,691 rows

Export complete: ~\Downloads\CFPB_Banking_Project\cfpb_banking_complaints_clean_2021_2025.csv


In [2]:
# Count the exported CSV rows in chunks to confirm that
# the file contains the same number of records as MySQL.
csv_row_count = sum(
    len(chunk)
    for chunk in pd.read_csv(
        output_path,
        chunksize=50_000,
        low_memory=False
    )
)

# Display the row count and file size.
file_size_mb = output_path.stat().st_size / (1024 ** 2)

print(f"CSV rows: {csv_row_count:,}")
print(f"File size: {file_size_mb:,.1f} MB")

CSV rows: 888,691
File size: 295.5 MB
